## Score: 0.80881

## 1. Импорт библиотек и настройка

In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor, Pool

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

## 2. Загрузка данных

In [2]:
transaction_file = '/kaggle/input/alfa-challenge/df_transaction.pa'
train_file = '/kaggle/input/alfa-challenge/train.pa'

df_transaction = pd.read_parquet(transaction_file)
train = pd.read_parquet(train_file)

print("Данные успешно загружены.")

Данные успешно загружены.


## 3. Подготовка данных

In [3]:
df_transaction['mcc_code'] = df_transaction['mcc_code'].astype(str)
df_transaction['merchant_name'] = df_transaction['merchant_name'].astype(str)
df_transaction['date_time'] = pd.to_datetime(df_transaction['date_time'])

print("Типы данных преобразованы.")

agg_features = df_transaction.groupby('client_num').agg({
    'amount': ['sum', 'mean', 'std'],
    'merchant_name': 'nunique',
    'mcc_code': 'nunique',
    'date_time': ['min', 'max', 'count']
})
agg_features.columns = ['_'.join(col).strip() for col in agg_features.columns.values]
agg_features.reset_index(inplace=True)

print("Агрегация признаков завершена.")

agg_features['date_time_min_hour'] = pd.to_datetime(agg_features['date_time_min'], unit='s').dt.hour
agg_features['date_time_min_day'] = pd.to_datetime(agg_features['date_time_min'], unit='s').dt.day
agg_features['date_time_min_month'] = pd.to_datetime(agg_features['date_time_min'], unit='s').dt.month
agg_features['date_time_max_hour'] = pd.to_datetime(agg_features['date_time_max'], unit='s').dt.hour
agg_features['date_time_max_day'] = pd.to_datetime(agg_features['date_time_max'], unit='s').dt.day
agg_features['date_time_max_month'] = pd.to_datetime(agg_features['date_time_max'], unit='s').dt.month

agg_features = agg_features.drop(columns=['date_time_min', 'date_time_max'])

print("Временные признаки подготовлены.")

train = train.merge(agg_features, on='client_num', how='left')
train.fillna({'amount_std': 0}, inplace=True)
print("Данные объединены и заполнены пропуски.")

target_column = 'target'
X = train.drop(columns=[target_column, 'client_num'])
y = train[target_column]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Данные разделены на обучающую и валидационную выборки.")

Типы данных преобразованы.
Агрегация признаков завершена.
Временные признаки подготовлены.
Данные объединены и заполнены пропуски.
Данные разделены на обучающую и валидационную выборки.


## 4. Расчет весов

In [4]:
unique_classes = np.sort(y.unique())
class_counts = y.value_counts()
total_samples = len(y)
class_weights = {c: total_samples / (len(unique_classes) * class_counts[c]) for c in unique_classes}
print("Веса классов рассчитаны.")

# Функция для маппинга весов
def map_class_weights(y, class_weights):
    return y.map(class_weights).values

weights_train = map_class_weights(y_train, class_weights)
weights_valid = map_class_weights(y_valid, class_weights)
print("Веса для выборок рассчитаны.")

Веса классов рассчитаны.
Веса для выборок рассчитаны.


## 5. Подготовка данных для CatBoost

In [5]:
train_pool = Pool(X_train, label=y_train, weight=weights_train)
valid_pool = Pool(X_valid, label=y_valid, weight=weights_valid)
print("Данные подготовлены для CatBoost.")

Данные подготовлены для CatBoost.


## 6. Определение кастомной метрики

In [6]:
class WMAEMetric:
    def get_final_error(self, error, weight):
        return error / weight

    def is_max_optimal(self):
        return False

    def evaluate(self, approxes, target, weight):
        approx = approxes[0]
        weight = np.ones_like(target) if weight is None else weight
        error = np.sum(weight * np.abs(target - approx))
        return error, np.sum(weight)
        
print("Кастомная метрика WMAE определена.")

Кастомная метрика WMAE определена.


## 7. Настройка параметров модели CatBoost

In [7]:
params = {
    'iterations': 1000,
    'eval_metric': WMAEMetric(),
    'loss_function': 'MAE',
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 50
}
print("Параметры модели настроены.")

Параметры модели настроены.


## 8. Обучение модели CatBoost

In [8]:
model = CatBoostRegressor(**params)
model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True,
    verbose=True
)
print("Модель обучена.")

y_valid_pred = model.predict(X_valid)
wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('Validation WMAE:', wmae)

0:	learn: 1.7081716	test: 1.7083266	best: 1.7083266 (0)	total: 1.8s	remaining: 30m 3s
1:	learn: 1.7005620	test: 1.7008054	best: 1.7008054 (1)	total: 1.82s	remaining: 15m 7s
2:	learn: 1.6960011	test: 1.6961075	best: 1.6961075 (2)	total: 1.83s	remaining: 10m 8s
3:	learn: 1.6900400	test: 1.6902065	best: 1.6902065 (3)	total: 1.84s	remaining: 7m 38s
4:	learn: 1.6856974	test: 1.6859576	best: 1.6859576 (4)	total: 1.85s	remaining: 6m 8s
5:	learn: 1.6783071	test: 1.6784228	best: 1.6784228 (5)	total: 1.86s	remaining: 5m 8s
6:	learn: 1.6731601	test: 1.6734550	best: 1.6734550 (6)	total: 1.88s	remaining: 4m 26s
7:	learn: 1.6690771	test: 1.6695037	best: 1.6695037 (7)	total: 1.89s	remaining: 3m 54s
8:	learn: 1.6635652	test: 1.6640427	best: 1.6640427 (8)	total: 1.9s	remaining: 3m 29s
9:	learn: 1.6583337	test: 1.6589582	best: 1.6589582 (9)	total: 1.91s	remaining: 3m 9s
10:	learn: 1.6531887	test: 1.6537790	best: 1.6537790 (10)	total: 1.92s	remaining: 2m 52s
11:	learn: 1.6485282	test: 1.6492143	best: 1.6

## 9. Количество уникальных клиентов

#### Поскольку в условиях хакатона не уточнялось, как формируется тестовая выборка, я решил исследовать данные. Для этого я подсчитал количество уникальных клиентов в файлах df_transaction и train. На основе этого анализа пришел к выводу, что тестовая выборка состоит из клиентов, которые присутствуют в df_transaction, но отсутствуют в train.

In [9]:
unique_clients_transaction = df_transaction['client_num'].nunique()
unique_clients_train = train['client_num'].nunique()

print(f"Количество уникальных client_num в df_transaction.pa: {unique_clients_transaction}")
print(f"Количество уникальных client_num в train.pa: {unique_clients_train}")

Количество уникальных client_num в df_transaction.pa: 109143
Количество уникальных client_num в train.pa: 70000


## 10. Сохранение результатов и создание submission

In [10]:
remaining_clients = agg_features[~agg_features['client_num'].isin(train['client_num'])]

X_remaining = remaining_clients.drop(columns=['client_num'])

remaining_clients['target'] = model.predict(X_remaining)

remaining_clients[['client_num', 'target']].to_csv('test.csv', index=False)
print("Результаты сохранены в test.csv")

remaining_clients[['client_num', 'target']].head(5)

Результаты сохранены в test.csv


,client_num,target
0,0,2.248551
10,10,2.802240
11,11,1.182098
14,14,4.060387
16,16,2.716566
